# Virtual Mouse / Air Canvas - Phase 8

## Air-writing sentences (letters + digits, confirm each character)

Write characters in the air and build up words and sentences. Each character
is confirmed with a gesture - the most reliable approach (it avoids the very
hard problem of automatically segmenting joined letters).

| Gesture | Action |
|---------|--------|
| **Index finger** (multi-stroke ok) | draw the current character |
| **Open palm** | recognise the character, append to the sentence |
| **Pinch** (index+thumb) | add a space |
| **Two fingers up** | backspace (delete last character) |
| **Fist** | clear the current drawing (re-do this character) |

Still scikit-learn (coexists with MediaPipe, no TensorFlow conflict). The model
now uses **EMNIST** (letters + digits) instead of MNIST (digits only).

## 1. Install (run once)
`emnist` downloads the dataset; sklearn/joblib you already have.

In [ ]:
!pip install emnist scikit-learn joblib pandas

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)

   ---------------------------------------- 0/6 [urllib3]
   ------ --------------------------------- 1/6 [tqdm]
   ------------- -------------------------- 2/6 [idna]
   -------------------- ------------------- 3/6 [charset_normalizer]
   -------------------------- ------------- 4/6 [requests]
   -------------------------- ------------- 4/6 [requests]
   ---------------------------------------- 6/6 [emnist]



## 2. Train the character model

Uses the EMNIST **balanced** set: 47 classes (10 digits + 26 uppercase + 11
distinct lowercase). First run downloads ~500 MB and training takes longer than
MNIST (a few minutes on your i7). Saves `char_model.joblib` and the label map.

> EMNIST images come in rotated+flipped; the code corrects orientation so they
> match how you draw. This is a classic EMNIST gotcha - handled here.

In [ ]:
import numpy as np
from emnist import extract_training_samples, extract_test_samples
from sklearn.neural_network import MLPClassifier
import joblib

# EMNIST balanced label -> character (standard mapping)
LABEL_MAP = ("0123456789"
             "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
             "abdefghnqrt")
assert len(LABEL_MAP) == 47

print("Loading EMNIST balanced (first run downloads ~500 MB)...")
X_tr, y_tr = extract_training_samples("balanced")
X_te, y_te = extract_test_samples("balanced")

# EMNIST images are stored transposed; fix orientation so they look upright.
X_tr = np.array([np.fliplr(np.rot90(img, 3)) for img in X_tr])
X_te = np.array([np.fliplr(np.rot90(img, 3)) for img in X_te])

# flatten + normalise
X_tr = X_tr.reshape(len(X_tr), -1).astype(np.float32) / 255.0
X_te = X_te.reshape(len(X_te), -1).astype(np.float32) / 255.0
print("Train:", X_tr.shape, "Test:", X_te.shape)

print("Training MLP (this takes a few minutes)...")
clf = MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=40,
                    early_stopping=True, random_state=42, verbose=False)
clf.fit(X_tr, y_tr)

acc = clf.score(X_te, y_te)
print(f"Test accuracy: {acc:.3f}")

joblib.dump({"model": clf, "label_map": LABEL_MAP}, "char_model.joblib")
print("Saved char_model.joblib")

Loading EMNIST balanced (first run downloads ~500 MB)...


BadZipFile: File is not a zip file

## 3. Sanity-check the saved model

In [ ]:
bundle = joblib.load("char_model.joblib")
clf2, lmap = bundle["model"], bundle["label_map"]
preds = clf2.predict(X_te[:8])
print("Predicted chars:", [lmap[p] for p in preds])
print("Actual chars   :", [lmap[t] for t in y_te[:8]])

## 4. Write the sentence air-writer app

Writes `sentence_writer.py`. Run from the terminal:

```
python sentence_writer.py
```

Write each character with your index finger, then:
**open palm** = recognise+append, **pinch** = space, **two fingers** = backspace,
**fist** = clear current drawing. The sentence shows on screen and is typed as
you confirm each character. Press **`q`** to quit.

In [ ]:
script = r"""
# sentence_writer.py - build sentences in the air, one confirmed character at a time.
import time, platform
import cv2
import numpy as np
import mediapipe as mp
import joblib
import pyautogui

try:
    bundle = joblib.load("char_model.joblib")
    MODEL = bundle["model"]
    LABEL_MAP = bundle["label_map"]
except Exception as e:
    raise SystemExit("Could not load char_model.joblib - run the training "
                     "notebook first. Error: " + str(e))

WRIST = 0
THUMB_TIP, INDEX_TIP, MIDDLE_TIP = 4, 8, 12
TIPS = [4, 8, 12, 16, 20]
PIPS = [3, 6, 10, 14, 18]
import math

def fingers_up(lm):
    f = [1 if lm[THUMB_TIP].x < lm[THUMB_TIP - 1].x else 0]
    for tip, pip in zip(TIPS[1:], PIPS[1:]):
        f.append(1 if lm[tip].y < lm[pip].y else 0)
    return f

def hand_size(lm):
    return math.hypot(lm[5].x - lm[WRIST].x, lm[5].y - lm[WRIST].y) + 1e-6

def pinch_ratio(lm):
    d = math.hypot(lm[INDEX_TIP].x - lm[THUMB_TIP].x,
                   lm[INDEX_TIP].y - lm[THUMB_TIP].y)
    return d / hand_size(lm)

def preprocess(canvas):
    ys, xs = np.where(canvas > 0)
    if len(xs) == 0:
        return None
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    crop = canvas[y0:y1+1, x0:x1+1]
    h, w = crop.shape
    side = max(h, w)
    sq = np.zeros((side, side), np.uint8)
    sq[(side-h)//2:(side-h)//2+h, (side-w)//2:(side-w)//2+w] = crop
    small = cv2.resize(sq, (20, 20), interpolation=cv2.INTER_AREA)
    img = np.zeros((28, 28), np.uint8)
    img[4:24, 4:24] = small
    return (img.astype(np.float32) / 255.0).reshape(1, 784)

def main():
    cap = cv2.VideoCapture(0, cv2.CAP_DSHOW) if platform.system()=="Windows" else cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    hands = mp.solutions.hands.Hands(static_image_mode=False, max_num_hands=1,
                                     min_detection_confidence=0.7, min_tracking_confidence=0.5)
    draw_util = mp.solutions.drawing_utils

    canvas = None
    prev_pt = None
    sentence = ""
    cooldown = 0
    COOLDOWN_FRAMES = 18

    print("Index=draw | palm=recognise | pinch=space | two=backspace | fist=clear | q=quit")
    while True:
        ok, frame = cap.read()
        if not ok:
            continue
        frame = cv2.flip(frame, 1)
        h, w = frame.shape[:2]
        if canvas is None:
            canvas = np.zeros((h, w), np.uint8)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        rgb.flags.writeable = False
        res = hands.process(rgb)
        rgb.flags.writeable = True

        status = "show hand"
        if cooldown > 0:
            cooldown -= 1

        if res.multi_hand_landmarks:
            hand = res.multi_hand_landmarks[0]
            draw_util.draw_landmarks(frame, hand, mp.solutions.hands.HAND_CONNECTIONS)
            lm = hand.landmark
            f = fingers_up(lm)
            total = sum(f)
            ratio = pinch_ratio(lm)

            if total == 0:                       # fist -> clear current char
                canvas[:] = 0; prev_pt = None
                status = "CLEARED char"
            elif ratio < 0.18 and cooldown == 0: # pinch -> space
                sentence += " "
                pyautogui.typewrite(" ")
                cooldown = COOLDOWN_FRAMES
                status = "SPACE"
            elif total == 2 and f[1] == 1 and f[2] == 1 and cooldown == 0:  # two -> backspace
                if sentence:
                    sentence = sentence[:-1]
                    pyautogui.press("backspace")
                cooldown = COOLDOWN_FRAMES
                status = "BACKSPACE"
            elif total == 5 and cooldown == 0:   # palm -> recognise+append
                vec = preprocess(canvas)
                if vec is not None:
                    pred = int(MODEL.predict(vec)[0])
                    ch = LABEL_MAP[pred]
                    sentence += ch
                    pyautogui.typewrite(ch)
                    canvas[:] = 0; prev_pt = None
                    cooldown = COOLDOWN_FRAMES
                    status = f"ADDED: {ch}"
                else:
                    status = "nothing drawn"
            else:
                index_only = f[1] == 1 and f[2] == 0 and ratio >= 0.18
                if index_only:
                    cx, cy = int(lm[INDEX_TIP].x*w), int(lm[INDEX_TIP].y*h)
                    if prev_pt is not None:
                        cv2.line(canvas, prev_pt, (cx, cy), 255, 16)
                    prev_pt = (cx, cy)
                    status = "DRAWING"
                else:
                    prev_pt = None

        # overlay strokes
        colour = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
        colour[:, :, 0] = 0
        frame = cv2.addWeighted(frame, 1.0, colour, 0.8, 0)

        cv2.putText(frame, status, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
        # sentence bar at the bottom
        cv2.rectangle(frame, (0, h-40), (w, h), (40,40,40), -1)
        cv2.putText(frame, "> " + sentence[-40:], (10, h-12),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,255), 2)

        cv2.imshow("Sentence Air Writer", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release(); cv2.destroyAllWindows(); hands.close()
    print("Final sentence:", sentence)

if __name__ == "__main__":
    main()
"""
with open("sentence_writer.py", "w") as fh:
    fh.write(script)
print("Wrote sentence_writer.py - run with: python sentence_writer.py")

## 5. Run & tips

```
python sentence_writer.py
```

### Getting good results
- Draw each character **large and centred** in the camera view.
- A character can be multiple strokes - the canvas holds them until you confirm.
- Open palm only when the character is finished.
- EMNIST merges look-alike cases (e.g. lowercase c/o/s map to uppercase), so
  some letters always come out uppercase - that's expected.

### Honest accuracy note
Letter recognition from air-drawing is harder than digits - expect more
mistakes, especially on letters that look similar (O/0, I/1/l, S/5). Backspace
is your friend. Drawing slowly and clearly helps a lot.

### Possible upgrades
- A dictionary/autocorrect pass to fix near-miss words.
- Confidence threshold: if the model is unsure, flash a warning instead of typing.
- Switch EMNIST 'balanced' -> 'byclass' for full upper+lower (62 classes, lower
  accuracy).